In [ ]:
we will load all the dependencies
we will setup openai and email
we will setup tool schema
we will define tool functions
we will define the tools schema
we will define the agent and validate the function arguments with pydantic tool schema
we will call main function

In [1]:
! pip install openai requests beautifulsoup4 pydantic python-dotenv rich

In [4]:
import os
import json
import base64
import requests
from bs4 import BeautifulSoup
from email.mime.text import MIMEText
from typing import List
import smtplib, ssl

from pydantic import BaseModel, Field
from openai import OpenAI
from google.colab import userdata

#openai and email setup
OPENAI_API_KEY = userdata.get("raj_api_key")
SENDER_EMAIL = userdata.get("email_id")
SENDER_PASSWORD = userdata.get("email_password")
client = OpenAI(api_key=OPENAI_API_KEY)


#tool schema
class NewsInput(BaseModel):
    topic: str = Field(..., description="Topic to search news for")


class EmailInput(BaseModel):
    to: str = Field(..., description="Recipient email")
    subject: str = Field(..., description="Email subject")
    message_text: str = Field(..., description="Email body content")


#Tool 1 web scraper
def get_news(topic: str) -> List[str]:
    url = f"https://news.google.com/search?q={topic}&hl=en-IN&gl=IN&ceid=IN:en"
    response = requests.get(url)

    soup = BeautifulSoup(response.text, "html.parser")

    headlines = [item.text for item in soup.select("h3")[:5]]
    return headlines


#Tool 2 email sender
def send_email(to: str, subject: str, message_text: str):
    port = 465
    smtp_server = "smtp.gmail.com"
    sender_email = SENDER_EMAIL
    password = SENDER_PASSWORD

    context = ssl.create_default_context()

    try:
        with smtplib.SMTP_SSL(smtp_server, port, context=context) as server:
            server.login(sender_email, password)
            message = f"Subject: {subject}\n\n{message_text}"
            server.sendmail(sender_email, to, message)
        return "Email sent successfully via SMTP!"
    except Exception as e:
        return f"Error sending email via SMTP: {e}"


#Tool defination
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_news",
            "description": "Fetch latest news headlines",
            "parameters": NewsInput.model_json_schema()
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send email using SMTP",
            "parameters": EmailInput.model_json_schema()
        }
    }
]


#Agent calling
def run_agent(user_prompt):
    messages = [
        {
            "role": "system",
            "content": """
You are an autonomous AI agent.

Follow these rules:
1. Always fetch news first
2. Then summarize clearly
3. Then send email
4. Never skip steps

Also include the impact on the stock market due to fetched the news and show the dependencies on different sectors of the economy.
"""
        },
        {"role": "user", "content": user_prompt}
    ]

    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if msg.tool_calls:
            for tool_call in msg.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                print(f"\n Tool Call: {name} → {args}")

                # Validating with Pydantic
                if name == "get_news":
                    validated = NewsInput(**args)
                    result = get_news(**validated.model_dump())

                elif name == "send_email":
                    validated = EmailInput(**args)
                    result = send_email(**validated.model_dump())

                messages.append(msg)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })

        else:
            print("\n FINAL RESPONSE:\n")
            print(msg.content)
            break


#main function
if __name__ == "__main__":
    run_agent(
        input("enter the latest news topic and email id: ")
    )

enter the latest news topic and email id: news on cricket and send it to rajwardhan828@gmail.com

 Tool Call: get_news → {'topic': 'cricket'}

 Tool Call: send_email → {'to': 'rajwardhan828@gmail.com', 'subject': 'Latest Cricket News Update', 'message_text': "Here is the summary of the latest cricket news:\n\n1. **LSG vs IPL vs MI**: The news is focused around LSG, IPL, and MI, possibly discussing recent match outcomes, strategies, or player performances.\n\n2. **PZ Cricket**: There may be developments or news about the Peshawar Zalmi team or events related to cricket in Pakistan.\n\n3. **GT vs IPL vs PBKS**: Updates might involve Gujarat Titans (GT), their participation in IPL, and interactions or matches against Punjab Kings (PBKS).\n\nThese pieces of news could indirectly impact the stock market, particularly the sectors connected to sports merchandise, broadcasting rights, and sponsorship industries. For example:\n\n- **Broadcasting Companies**: Companies with broadcasting rights m